In [1]:
# ======================
# Imports e configuração
# ======================
import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv

import pandas as pd
import numpy as np

from bbmagic import Db2

# Utilitários adicionais
import csv
from typing import Iterable, Optional, List, Dict

In [2]:
# =========================================
# Conexão DB2 (variáveis no arquivo conexao.env)
# =========================================
# Estrutura esperada do conexao.env:
load_dotenv(dotenv_path='conexao.env')

db2_user = os.getenv('DB2_USER')
db2_password = os.getenv('DB2_PASSWORD')

# Observação: a classe Db2 do bbmagic geralmente aceita chaves do dicionário/env
# Se a sua instância exige parâmetros nomeados, ajuste conforme o seu ambiente.
db2 = Db2('db2_user','db2_password')

In [3]:
sql = """
SELECT DISTINCT
       AA_CPTC
FROM DB2CCA.TTL_DSMP_CMBL_BC
ORDER BY AA_CPTC
"""

df_anos_disponiveis = db2.query(sql, chunksize=None)
df_anos_disponiveis

,AA_CPTC
0,2016
1,2017
2,2018
3,2019
4,2020
5,2021
6,2022
7,2023
8,2024
9,2025


In [4]:
# ====================================================================
# Consulta 1: BC (mercado financeiro) agregado anual + mapping cliente
# ====================================================================
sql = """
    WITH CLIENTES_TRANSFORMADOS AS (
        SELECT 
            COD,
            CAST(COD_CPF_CGC AS VARCHAR(20)) AS COD_CPF_CGC_ORIGINAL,
            CASE
                WHEN LENGTH(CAST(COD_CPF_CGC AS VARCHAR(20))) > 6 THEN
                    LEFT(CAST(COD_CPF_CGC AS VARCHAR(20)), LENGTH(CAST(COD_CPF_CGC AS VARCHAR(20))) - 6)
                ELSE CAST(COD_CPF_CGC AS VARCHAR(20))
            END AS NR_CPF_CNPJ_CLI,  -- radical/base do CNPJ
            NOM
        FROM DB2MCI.CLIENTE
        WHERE COD_TIPO = 2
    ),
    CLIENTE_MINIMO AS (
        SELECT
            NR_CPF_CNPJ_CLI,
            COD_CPF_CGC_ORIGINAL,
            NOM,
            COD,
            ROW_NUMBER() OVER (
                PARTITION BY NR_CPF_CNPJ_CLI
                ORDER BY CAST(COD_CPF_CGC_ORIGINAL AS BIGINT) ASC
            ) AS rn
        FROM CLIENTES_TRANSFORMADOS
        WHERE NR_CPF_CNPJ_CLI IS NOT NULL
    ),
    BC_AGG AS (
        SELECT
            AA_CPTC, -- Ano
            NR_CPF_CNPJ_CLI, -- Radical CNPJ
            CD_TIP_OPR, -- Tipo de operação
            SUM(VL_TTL_CTRD)/1000 AS valor_contratado_sfn,
            SUM(VL_TTL_LQDD)/1000 AS valor_liquidado_sfn,
            SUM(VL_TTL_CNCD)/1000 AS valor_cancelado_sfn,
            SUM(VL_TTL_BXAD)/1000 AS valor_baixado_sfn
        FROM DB2CCA.TTL_DSMP_CMBL_BC
        WHERE AA_CPTC IN (2024, 2025, 2026)
        AND CD_TIP_OPR IN (1, 2)
        AND CD_TIP_PSS_CLI = 2
        AND NR_CPF_CNPJ_CLI IS NOT NULL
        AND NR_CPF_CNPJ_CLI <> 0
        GROUP BY AA_CPTC, NR_CPF_CNPJ_CLI, CD_TIP_OPR
    )
    SELECT
        b.AA_CPTC,
        b.CD_TIP_OPR,
        b.NR_CPF_CNPJ_CLI,
        c.COD,
        c.NOM,
        b.valor_contratado_sfn AS VALOR_CONTRATADO_SFN,
        b.valor_liquidado_sfn  AS VALOR_LIQUIDADO_SFN,
        b.valor_cancelado_sfn  AS VALOR_CANCELADO_SFN,
        b.valor_baixado_sfn    AS VALOR_BAIXADO_SFN
    FROM BC_AGG b
    JOIN CLIENTE_MINIMO c
      ON CAST(b.NR_CPF_CNPJ_CLI AS VARCHAR(20)) = c.NR_CPF_CNPJ_CLI
     AND c.rn = 1
    ORDER BY b.AA_CPTC, b.NR_CPF_CNPJ_CLI, b.CD_TIP_OPR
"""
df_TTL_DSMP_CMBL_BC_anual = db2.query(sql, chunksize=None)
df_TTL_DSMP_CMBL_BC_anual

,AA_CPTC,CD_TIP_OPR,NR_CPF_CNPJ_CLI,COD,NOM,VALOR_CONTRATADO_SFN,VALOR_LIQUIDADO_SFN,VALOR_CANCELADO_SFN,VALOR_BAIXADO_SFN
0,2024,2,410,931960748,BIOAGRI ANALISES DE ALIMENTOS LTDA. ...,134.74,134.74,0.0,0.0
1,2024,2,1180,300876799,CENTRAIS ELETRICAS BRASILEIRAS S/A-ELETROBRAS ...,26513.08,26513.08,0.0,0.0
2,2024,2,1596,36229862,OLITRACTOR IMPORTACAO E EXPORTACAO LTDA ...,1411.64,1411.64,0.0,0.0
3,2024,1,3516,111559863,MC BAUCHEMIE BRASIL INDUSTRIA E COMERCIO LTDA ...,268.77,268.77,0.0,0.0
4,2024,2,4559,901607643,JUMORI DISTRIBUIDORA DE AUTO PECAS LTDA ...,435.31,435.31,0.0,0.0
...,...,...,...,...,...,...,...,...,...
68347,2026,2,97493373,203952212,MALINSKI MADEIRAS LTDA ...,91.64,91.64,0.0,0.0
68348,2026,1,97755177,400814556,A. GRINGS S/A ...,3580.68,3580.68,0.0,0.0
68349,2026,2,97755177,400814556,A. GRINGS S/A ...,3.55,3.55,0.0,0.0
68350,2026,1,97762868,400814744,CALCADOS TABITA LTDA ...,284.65,440.14,0.0,0.0


In [5]:
# ===============================================
# Consulta 2: BB agregado anual + mapping cliente
# ===============================================
sql = """
    WITH CLIENTES_TRANSFORMADOS AS (
        SELECT 
            COD,
            CAST(COD_CPF_CGC AS VARCHAR(20)) AS COD_CPF_CGC_ORIGINAL,
            CASE
                WHEN LENGTH(CAST(COD_CPF_CGC AS VARCHAR(20))) > 6 THEN
                    LEFT(CAST(COD_CPF_CGC AS VARCHAR(20)), LENGTH(CAST(COD_CPF_CGC AS VARCHAR(20))) - 6)
                ELSE CAST(COD_CPF_CGC AS VARCHAR(20))
            END AS NR_CPF_CNPJ_CLI,  -- radical/base do CNPJ
            NOM
        FROM DB2MCI.CLIENTE
        WHERE COD_TIPO = 2
    ),
    CLIENTE_MINIMO AS (
        SELECT
            NR_CPF_CNPJ_CLI,
            COD_CPF_CGC_ORIGINAL,
            NOM,
            COD,
            ROW_NUMBER() OVER (
                PARTITION BY NR_CPF_CNPJ_CLI
                ORDER BY CAST(COD_CPF_CGC_ORIGINAL AS BIGINT) ASC
            ) AS rn
        FROM CLIENTES_TRANSFORMADOS
        WHERE NR_CPF_CNPJ_CLI IS NOT NULL
    ),
    BB_AGG AS (
        SELECT
            AA_CPTC,
            NR_CPF_CNPJ_CLI,
            CD_TIP_OPR,
            SUM(VL_TTL_CTRD)/1000 AS valor_contratado_bb,
            SUM(VL_TTL_LQDD)/1000 AS valor_liquidado_bb,
            SUM(VL_TTL_CNCD)/1000 AS valor_cancelado_bb,
            SUM(VL_TTL_BXAD)/1000 AS valor_baixado_bb
        FROM DB2CCA.TTL_DSMP_CMBL_BB
        WHERE AA_CPTC IN (2024, 2025, 2026)
        AND CD_TIP_OPR IN (1, 2)
        AND CD_TIP_PSS_CLI = 2
        AND NR_CPF_CNPJ_CLI IS NOT NULL
        AND NR_CPF_CNPJ_CLI <> 0
        GROUP BY AA_CPTC, NR_CPF_CNPJ_CLI, CD_TIP_OPR
    )
    SELECT
        b.AA_CPTC,
        b.CD_TIP_OPR,
        b.NR_CPF_CNPJ_CLI,
        c.COD,
        c.NOM,
        b.valor_contratado_bb AS VALOR_CONTRATADO_BB,
        b.valor_liquidado_bb  AS VALOR_LIQUIDADO_BB,
        b.valor_cancelado_bb  AS VALOR_CANCELADO_BB,
        b.valor_baixado_bb    AS VALOR_BAIXADO_BB
    FROM BB_AGG b
    JOIN CLIENTE_MINIMO c
      ON CAST(b.NR_CPF_CNPJ_CLI AS VARCHAR(20)) = c.NR_CPF_CNPJ_CLI
     AND c.rn = 1
    ORDER BY b.AA_CPTC, b.NR_CPF_CNPJ_CLI, b.CD_TIP_OPR
"""
df_TTL_DSMP_CMBL_BB_anual = db2.query(sql, chunksize=None)
df_TTL_DSMP_CMBL_BB_anual

,AA_CPTC,CD_TIP_OPR,NR_CPF_CNPJ_CLI,COD,NOM,VALOR_CONTRATADO_BB,VALOR_LIQUIDADO_BB,VALOR_CANCELADO_BB,VALOR_BAIXADO_BB
0,2024,2,1180,300876799,CENTRAIS ELETRICAS BRASILEIRAS S/A-ELETROBRAS ...,26.33,26.33,0.0,0.0
1,2024,1,1564,203901796,CIPOLATTI & CIPOLATTI EXPORTACAO E IMPORTACAO ...,114.27,114.27,0.0,0.0
2,2024,2,1596,36229862,OLITRACTOR IMPORTACAO E EXPORTACAO LTDA ...,1411.64,1411.64,0.0,0.0
3,2024,2,5063,30221952,INSTRUVAL INSTRUMENTOS E SERVICOS LTDA ...,205.63,205.63,0.0,0.0
4,2024,2,6027,921048880,SERQUIMICA INDUSTRIA COMERCIO IMP EXP PRODS QU...,4079.08,4079.08,0.0,0.0
...,...,...,...,...,...,...,...,...,...
51919,2026,1,97837181,905941901,DEXCO S.A. ...,3000.99,3000.99,0.0,0.0
51920,2026,1,97969810,400816259,WERNER CALCADOS LTDA ...,284.02,284.02,0.0,0.0
51921,2026,1,98521909,400027750,VINICOLA CAMPESTRE LTDA ...,72.11,72.11,0.0,0.0
51922,2026,2,98521909,400027750,VINICOLA CAMPESTRE LTDA ...,393.62,392.83,0.0,0.0


In [6]:
# ===========================================================
# Consulta 3: Período mensal por radical de CNPJ (INICIO/FIM)
# ===========================================================
sql = """
    WITH bc AS (
        SELECT AA_CPTC, MM_CPTC, NR_CPF_CNPJ_CLI, CD_TIP_OPR
        FROM DB2CCA.TTL_DSMP_CMBL_BC
        WHERE AA_CPTC IN (2024, 2025, 2026)
        AND CD_TIP_OPR IN (1, 2)
        AND CD_TIP_PSS_CLI = 2
        AND NR_CPF_CNPJ_CLI IS NOT NULL
        AND NR_CPF_CNPJ_CLI <> 0
        AND MM_CPTC BETWEEN 1 AND 12
    ),
    bb AS (
        SELECT AA_CPTC, MM_CPTC, NR_CPF_CNPJ_CLI, CD_TIP_OPR
        FROM DB2CCA.TTL_DSMP_CMBL_BB
        WHERE AA_CPTC IN (2024, 2025, 2026)
        AND CD_TIP_OPR IN (1, 2)
        AND CD_TIP_PSS_CLI = 2
        AND NR_CPF_CNPJ_CLI IS NOT NULL
        AND NR_CPF_CNPJ_CLI <> 0
        AND MM_CPTC BETWEEN 1 AND 12
    ),
    base AS (
        SELECT DISTINCT AA_CPTC, MM_CPTC, NR_CPF_CNPJ_CLI
        FROM (
            SELECT AA_CPTC, MM_CPTC, NR_CPF_CNPJ_CLI FROM bc
            UNION ALL
            SELECT AA_CPTC, MM_CPTC, NR_CPF_CNPJ_CLI FROM bb
        ) t
    ),
    base_dates AS (
        SELECT
            NR_CPF_CNPJ_CLI,
            DATE(
                TIMESTAMP_FORMAT(
                    CHAR(AA_CPTC) || '-' || RIGHT('00' || RTRIM(CHAR(MM_CPTC)), 2) || '-01',
                    'YYYY-MM-DD'
                )
            ) AS dt
        FROM base
    ),
    agg AS (
        SELECT
            NR_CPF_CNPJ_CLI,
            MIN(dt) AS dt_inicio,
            MAX(dt) AS dt_fim
        FROM base_dates
        GROUP BY NR_CPF_CNPJ_CLI
    ),
    rotulos AS (
        SELECT
            NR_CPF_CNPJ_CLI,
            CASE MONTH(dt_inicio)
                WHEN 1  THEN 'jan' WHEN 2  THEN 'fev' WHEN 3  THEN 'mar' WHEN 4  THEN 'abr'
                WHEN 5  THEN 'mai' WHEN 6  THEN 'jun' WHEN 7  THEN 'jul' WHEN 8  THEN 'ago'
                WHEN 9  THEN 'set' WHEN 10 THEN 'out' WHEN 11 THEN 'nov' WHEN 12 THEN 'dez'
            END || '/' || CHAR(YEAR(dt_inicio)) AS inicio,
            CASE MONTH(dt_fim)
                WHEN 1  THEN 'jan' WHEN 2  THEN 'fev' WHEN 3  THEN 'mar' WHEN 4  THEN 'abr'
                WHEN 5  THEN 'mai' WHEN 6  THEN 'jun' WHEN 7  THEN 'jul' WHEN 8  THEN 'ago'
                WHEN 9  THEN 'set' WHEN 10 THEN 'out' WHEN 11 THEN 'nov' WHEN 12 THEN 'dez'
            END || '/' || CHAR(YEAR(dt_fim)) AS fim
        FROM agg
    )
    SELECT
        NR_CPF_CNPJ_CLI,
        inicio AS INICIO,
        fim    AS FIM
    FROM rotulos
    ORDER BY NR_CPF_CNPJ_CLI
"""
df_TTL_DSMP_CMBL_mensal = db2.query(sql, chunksize=None)
df_TTL_DSMP_CMBL_mensal

,NR_CPF_CNPJ_CLI,INICIO,FIM
0,410,fev/2024,dez/2025
1,1180,jul/2024,dez/2025
2,1564,jun/2024,jul/2025
3,1596,jan/2024,fev/2026
4,3516,fev/2024,dez/2025
...,...,...,...
32009,98521909,jan/2024,fev/2026
32010,98586662,jan/2024,out/2025
32011,98669997,jan/2024,set/2025
32012,98670144,jan/2024,fev/2026


In [7]:
# =========================================
# Merge: consolidação BC + BB + Período
# =========================================
df_performance_internacional = pd.merge(
    df_TTL_DSMP_CMBL_BC_anual,
    df_TTL_DSMP_CMBL_BB_anual,
    on=['AA_CPTC', 'CD_TIP_OPR', 'COD', 'NOM', 'NR_CPF_CNPJ_CLI'],
    how='outer',
)

df_performance_internacional = pd.merge(
    df_performance_internacional,
    df_TTL_DSMP_CMBL_mensal,
    on=['NR_CPF_CNPJ_CLI'],
    how='left',
)

# Para o relatório final não precisamos mais do radical
df_performance_internacional.drop(columns=['NR_CPF_CNPJ_CLI'], inplace=True)
df_performance_internacional.head()

,AA_CPTC,CD_TIP_OPR,COD,NOM,VALOR_CONTRATADO_SFN,VALOR_LIQUIDADO_SFN,VALOR_CANCELADO_SFN,VALOR_BAIXADO_SFN,VALOR_CONTRATADO_BB,VALOR_LIQUIDADO_BB,VALOR_CANCELADO_BB,VALOR_BAIXADO_BB,INICIO,FIM
0,2024,2,931960748,BIOAGRI ANALISES DE ALIMENTOS LTDA. ...,134.74,134.74,0.0,0.0,NaN,NaN,NaN,NaN,fev/2024,dez/2025
1,2024,2,300876799,CENTRAIS ELETRICAS BRASILEIRAS S/A-ELETROBRAS ...,26513.08,26513.08,0.0,0.0,26.33,26.33,0.0,0.0,jul/2024,dez/2025
2,2024,2,36229862,OLITRACTOR IMPORTACAO E EXPORTACAO LTDA ...,1411.64,1411.64,0.0,0.0,1411.64,1411.64,0.0,0.0,jan/2024,fev/2026
3,2024,1,111559863,MC BAUCHEMIE BRASIL INDUSTRIA E COMERCIO LTDA ...,268.77,268.77,0.0,0.0,NaN,NaN,NaN,NaN,fev/2024,dez/2025
4,2024,2,901607643,JUMORI DISTRIBUIDORA DE AUTO PECAS LTDA ...,435.31,435.31,0.0,0.0,NaN,NaN,NaN,NaN,fev/2024,dez/2025


In [ ]:
# ===================================================
# Imports necessários
# ===================================================
import os
import pandas as pd
from typing import Iterable, Optional, List
import csv

from sqlalchemy import create_engine

# ===================================================
# Funções de relatório e limpeza
# ===================================================
def limpar_invisiveis(s: str) -> str:
    """Remove caracteres invisíveis comuns: BOM, zero-width, LRM, NBSP, tabs e normaliza quebras."""
    if not isinstance(s, str):
        return s
    for ch in ["\ufeff", "\u200b", "\u200e", "\u00a0"]:
        s = s.replace(ch, "")
    return s.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")

def limpar_primeira_linha(texto: str) -> str:
    """Remove espaços e invisíveis só da primeira linha (cabeçalho COD - NOM)."""
    if not isinstance(texto, str) or not texto:
        return texto
    linhas = texto.split("\n")
    l0 = limpar_invisiveis(linhas[0]).rstrip()
    linhas[0] = l0
    return "\n".join(linhas)


def gerar_relatorio_empresa(
    df: pd.DataFrame,
    cod_alvo: int,
    zerar_importacao: bool = False,
    fonte_label: str = "CCA"  # "UCE / CAM57",
) -> str:
    required_cols = [
        'COD', 'NOM', 'INICIO', 'FIM', 'AA_CPTC', 'CD_TIP_OPR',
        'VALOR_CONTRATADO_BB', 'VALOR_LIQUIDADO_BB',
        'VALOR_CONTRATADO_SFN', 'VALOR_LIQUIDADO_SFN'
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        return f"Colunas ausentes no DataFrame: {missing}"

    df_cod = df.loc[df['COD'].eq(cod_alvo)].copy()
    if df_cod.empty:
        return f"Nenhum registro encontrado para COD = {cod_alvo}."

    nome = limpar_invisiveis(str(df_cod['NOM'].iloc[0])).strip()

    # Normalização numérica
    for col in [
        'VALOR_CONTRATADO_BB', 'VALOR_LIQUIDADO_BB',
        'VALOR_CONTRATADO_SFN', 'VALOR_LIQUIDADO_SFN'
    ]:
        df_cod[col] = pd.to_numeric(df_cod[col], errors='coerce').fillna(0.0)

    df_cod['AA_CPTC'] = pd.to_numeric(df_cod['AA_CPTC'], errors='coerce').astype('Int64')

    # Parser pt-BR + ISO para INICIO/FIM
    meses_pt_br = {
        "jan":1, "fev":2, "mar":3, "abr":4, "mai":5, "jun":6,
        "jul":7, "ago":8, "set":9, "out":10, "nov":11, "dez":12
    }

    def parse_mes_ano(valor) -> pd.Timestamp:
        if pd.isna(valor):
            return pd.NaT
        if isinstance(valor, pd.Timestamp):
            return valor
        s = limpar_invisiveis(str(valor)).strip().lower()
        # Formato 'mmm/aaaa'
        if "/" in s and len(s) <= 8:
            try:
                mm_abbr, yyyy = s.split("/")
                mm = meses_pt_br.get(mm_abbr[:3])
                yyyy = int(yyyy)
                if mm is not None:
                    return pd.Timestamp(year=yyyy, month=mm, day=1)
            except Exception:
                pass
        return pd.to_datetime(s, errors='coerce')

    inicio_dt = df_cod['INICIO'].map(parse_mes_ano)
    fim_dt    = df_cod['FIM'].map(parse_mes_ano)

    dt_min_inicio = inicio_dt.min()
    dt_max_fim    = fim_dt.max()
    dt_max_inicio = inicio_dt.max()
    dt_max = dt_max_fim if pd.notna(dt_max_fim) else dt_max_inicio

    meses_pt_num = {1:"jan",2:"fev",3:"mar",4:"abr",5:"mai",6:"jun",7:"jul",8:"ago",9:"set",10:"out",11:"nov",12:"dez"}
    def fmt_mes_ano(dt: pd.Timestamp) -> str:
        return f"{meses_pt_num[dt.month]}/{dt.year}"

    if pd.notna(dt_min_inicio) and pd.notna(dt_max):
        if dt_min_inicio == dt_max:
            periodo_label = f"{fmt_mes_ano(dt_min_inicio)}"
        else:
            periodo_label = f"de {fmt_mes_ano(dt_min_inicio)} a {fmt_mes_ano(dt_max)}"
    elif pd.notna(dt_min_inicio) and pd.isna(dt_max):
        periodo_label = f"de {fmt_mes_ano(dt_min_inicio)} a (não informado)"
    elif pd.isna(dt_min_inicio) and pd.notna(dt_max):
        periodo_label = f"de (não informado) a {fmt_mes_ano(dt_max)}"
    else:
        periodo_label = "de (não informado) a (não informado)"

    # Agregação
    agg = (
        df_cod.groupby(['AA_CPTC', 'CD_TIP_OPR'], dropna=False, as_index=False)
        .agg({
            'VALOR_CONTRATADO_BB': 'sum',
            'VALOR_LIQUIDADO_BB': 'sum',
            'VALOR_CONTRATADO_SFN': 'sum',
            'VALOR_LIQUIDADO_SFN': 'sum'
        })
    )
    exp = agg.loc[agg['CD_TIP_OPR'].eq(1)].copy()
    imp = agg.loc[agg['CD_TIP_OPR'].eq(2)].copy()

    def fmt_int(x) -> str:
        try:
            s = f"{int(round(float(x))):,}"
        except Exception:
            s = "0"
        return s.replace(",", ".")

    def linhas_por_categoria(df_part: pd.DataFrame, exportacao: bool) -> List[str]:
        if df_part.empty:
            return []
        anos = sorted(df_part['AA_CPTC'].dropna().astype(int).unique().tolist())
        linhas: List[str] = []
        for ano in anos:
            row = df_part.loc[df_part['AA_CPTC'].eq(ano)]
            bb_c  = float(row['VALOR_CONTRATADO_BB'].sum())
            bb_l  = float(row['VALOR_LIQUIDADO_BB'].sum())
            sfn_c = float(row['VALOR_CONTRATADO_SFN'].sum())
            sfn_l = float(row['VALOR_LIQUIDADO_SFN'].sum())

            if (not exportacao) and zerar_importacao:
                bb_c = bb_l = sfn_c = sfn_l = 0.0

            linhas.append(
                f"{ano:<4}   {fmt_int(bb_c):>14}{fmt_int(bb_l):>12}{fmt_int(sfn_c):>14}{fmt_int(sfn_l):>12}"
            )
        return linhas

    cabecalho = (
        limpar_invisiveis(f"{cod_alvo} - {nome}").rstrip() + "\n"
        "-----------------------------------------------------------\n"
        ".              Banco do Brasil         Mercado Financeiro\n"
        "Período    Contratado   Liquidado    Contratado   Liquidado\n"
        "-----------------------------------------------------------\n"
        "Exportação (em US$ mil)\n"
        "-----------------------------------------------------------\n"
    )
    bloco_exp = "\n".join(linhas_por_categoria(exp, exportacao=True)) or "(sem registros)"

    separador = (
        "\n-----------------------------------------------------------\n"
        "Importação (em US$ mil)\n"
        "-----------------------------------------------------------\n"
    )
    bloco_imp = "\n".join(linhas_por_categoria(imp, exportacao=False)) or "(sem registros)"

    rodape = (
        "\n-----------------------------------------------------------\n"
        f"base dados:  {periodo_label}      Fonte: {fonte_label}"
    )

    texto = cabecalho + bloco_exp + separador + bloco_imp + rodape
    texto = limpar_primeira_linha(texto)
    return texto

In [20]:
import pandas as pd

df_relatorios = pd.DataFrame({
    "COD": df_performance_internacional["COD"].astype(int),
    "TEXTO": df_performance_internacional["COD"].astype(int).apply(
        lambda cod: gerar_relatorio_empresa(
            df=df_performance_internacional,
            cod_alvo=cod,
            zerar_importacao=False,  
            fonte_label="CCA"         
        )
    )
})

df_relatorios

,COD,TEXTO
0,931960748,931960748 - BIOAGRI ANALISES DE ALIMENTOS LTDA...
1,300876799,300876799 - CENTRAIS ELETRICAS BRASILEIRAS S/A...
2,36229862,36229862 - OLITRACTOR IMPORTACAO E EXPORTACAO ...
3,111559863,111559863 - MC BAUCHEMIE BRASIL INDUSTRIA E CO...
4,901607643,901607643 - JUMORI DISTRIBUIDORA DE AUTO PECAS...
...,...,...
83250,400367981,400367981 - MOINHO TAQUARIENSE LTDA\n---------...
83251,905941901,905941901 - DEXCO S.A.\n----------------------...
83252,400027750,400027750 - VINICOLA CAMPESTRE LTDA\n---------...
83253,400027750,400027750 - VINICOLA CAMPESTRE LTDA\n---------...


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# ===================================================
# 1) Carrega variáveis do postgresql.env
# ===================================================
load_dotenv("postgresql.env", override=True)

PG_HOST   = os.environ["PG_HOST"]
PG_PORT   = os.environ.get("PG_PORT", "5432")
PG_DB     = os.environ["PG_DATABASE"]
PG_USER   = os.environ["PG_USER"]
PG_PASS   = os.environ["PG_PASSWORD"]
PG_SCHEMA = os.environ.get("PG_SCHEMA", "public") 

TABELA = "performance_internacional"

# ===================================================
# 2) Cria engine PostgreSQL
# ===================================================
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

# ===================================================
# 3) DROP TABLE (equivalente ao PROC SQL do SAS)
# ===================================================
with engine.begin() as conn:
    conn.execute(
        text(f'DROP TABLE IF EXISTS "{PG_SCHEMA}"."{TABELA}"')
    )

# ===================================================
# 4) SUBIR a tabela
# ===================================================
df_relatorios.to_sql(
    name=TABELA,
    con=engine,
    schema=PG_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000
)

print(f"✅ Tabela {PG_SCHEMA}.{TABELA} recriada com {len(df_relatorios)} registros.")

OperationalError: (psycopg2.OperationalError) 
(Background on this error at: https://sqlalche.me/e/20/e3q8)